<a href="https://colab.research.google.com/github/keyvanmahmoudi/Data_Science_Course/blob/main/solutions/Session09_statistics_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۹: آمار برای علم داده

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session09_statistics.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

کتابخانه‌ها و داده‌هایی که پاسخ‌ها به آن‌ها نیاز دارند:

</div>

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

rng = np.random.default_rng(1405)
DATA_URL = "https://raw.githubusercontent.com/keyvanmahmoudi/Data_Science_Course/main/data/"
tips = pd.read_csv(DATA_URL + "tips.csv")
tips["tip_pct"] = tips["tip"] / tips["total_bill"] * 100
titanic = pd.read_csv(DATA_URL + "titanic.csv")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

بازهٔ اطمینان ۹۵٪ برای میانگین `total_bill` را با فرمول و با بوت‌استرپ حساب کنید و مقایسه کنید.

</div>

In [2]:
bill = tips["total_bill"].values
n = len(bill)
se = bill.std(ddof=1) / np.sqrt(n)
formula = stats.t.interval(0.95, df=n - 1, loc=bill.mean(), scale=se)
boot = np.array([rng.choice(bill, n).mean() for _ in range(10_000)])
print("formula  :", np.round(formula, 2))
print("bootstrap:", np.round(np.percentile(boot, [2.5, 97.5]), 2))

formula  : [18.66 20.91]
bootstrap: [18.67 20.9 ]


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

آیا درصد انعام در شام و ناهار متفاوت است؟ آزمون t و آزمون جایگشتی را انجام دهید و نتیجه را در یک جمله بنویسید.

</div>

In [3]:
dinner = tips.loc[tips["time"] == "Dinner", "tip_pct"].values
lunch = tips.loc[tips["time"] == "Lunch", "tip_pct"].values
t = stats.ttest_ind(dinner, lunch, equal_var=False)

observed = dinner.mean() - lunch.mean()
pooled = np.concatenate([dinner, lunch])
diffs = []
for _ in range(10_000):
    rng.shuffle(pooled)
    diffs.append(pooled[:len(dinner)].mean() - pooled[len(dinner):].mean())
p_perm = (np.abs(diffs) >= abs(observed)).mean()

print(f"dinner {dinner.mean():.2f}% vs lunch {lunch.mean():.2f}%  (difference {observed:.2f})")
print(f"Welch t-test p = {t.pvalue:.3f}, permutation p = {p_perm:.3f}")

dinner 15.95% vs lunch 16.41%  (difference -0.46)
Welch t-test p = 0.514, permutation p = 0.596


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

هر دو آزمون p-value بزرگی می‌دهند؛ **شواهدی** برای تفاوت درصد انعام در شام و ناهار نداریم.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

با آزمون کای‌دو بررسی کنید که آیا زنده ماندن در تایتانیک به کلاس سفر (`pclass`) وابسته است.

</div>

In [4]:
table = pd.crosstab(titanic["pclass"], titanic["survived"])
chi2, p, dof, _ = stats.chi2_contingency(table)
print(table, "\n")
print(titanic.groupby("pclass")["survived"].mean().round(3), "\n")
print(f"chi2 = {chi2:.1f}, dof = {dof}, p-value = {p:.1e}")

survived    0    1
pclass            
1          80  136
2          97   87
3         372  119 

pclass
1    0.630
2    0.473
3    0.242
Name: survived, dtype: float64 

chi2 = 102.9, dof = 2, p-value = 4.5e-23


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

آزمون A/B بالا را با ۵۰۰ کاربر در هر گروه (به‌جای ۴۰۰۰) دوباره شبیه‌سازی کنید. چند بار از ۱۰۰ بار اجرا، اثر واقعی را تشخیص می‌دهید؟ (این عدد، **توان** آزمون است.)

</div>

In [5]:
def ab_test(n_per_group, p_a=0.100, p_b=0.118):
    conv = np.array([(rng.random(n_per_group) < p_b).sum(), (rng.random(n_per_group) < p_a).sum()])
    _, p = proportions_ztest(conv, np.array([n_per_group, n_per_group]))
    return p < 0.05

for n in [500, 4000]:
    power = np.mean([ab_test(n) for _ in range(1000)])
    print(f"{n:5d} users per group: real effect detected in {power:.0%} of experiments")

  500 users per group: real effect detected in 15% of experiments


 4000 users per group: real effect detected in 72% of experiments


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

با ۵۰۰ کاربر در هر گروه، اثر واقعی فقط در کسر کوچکی از آزمایش‌ها دیده می‌شود (توان پایین)؛ با ۴۰۰۰ کاربر، در بیشتر آن‌ها.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۵

شبیه‌سازی کنید: اگر ۲۰ آزمون مستقل روی دادهٔ کاملاً تصادفی انجام دهیم، احتمال اینکه **حداقل یکی** معنی‌دار شود چقدر است؟ با مقدار نظری $1 - 0.95^{20}$ مقایسه کنید.

</div>

In [6]:
n_experiments, n_tests = 5000, 20
hits = 0
for _ in range(n_experiments):
    a = rng.normal(size=(n_tests, 30))
    b = rng.normal(size=(n_tests, 30))
    p = stats.ttest_ind(a, b, axis=1).pvalue
    hits += (p < 0.05).any()
print(f"simulated P(at least one 'significant') = {hits / n_experiments:.3f}")
print(f"theory 1 - 0.95^20                      = {1 - 0.95 ** 20:.3f}")

simulated P(at least one 'significant') = 0.632
theory 1 - 0.95^20                      = 0.642
